#Validações - Qualidade de Dados
---
Este notebook contém scripts em SQL em que foram realizadas análises acerca da qualidade dos dados, considerando os seguintes aspectos:  
1. Completude;  
3. Consistência;  
4. Unicidade;  
5. Acurácia;  
6. _Outliers_.  

## 1. Completude

In [0]:
SELECT
    COUNT(*) AS total_registros,
    COUNT(*) FILTER (WHERE co_catmat IS NULL) AS catmat_nulos,
    COUNT(*) FILTER (WHERE co_seq_bps IS NULL) AS cod_bps_nulos,
    COUNT(*) FILTER (WHERE dt_compra IS NULL) AS dt_compra_nulos,
    COUNT(*) FILTER (WHERE dt_insercao IS NULL) AS dt_insercao_nulos,
    COUNT(*) FILTER (WHERE modalidade IS NULL) AS modalidade_nulos,
    COUNT(*) FILTER (WHERE vl_preco_unitario IS NULL) AS preco_unitario_nulos,
    COUNT(*) FILTER (WHERE vl_preco_total IS NULL) AS preco_total_nulos,
    COUNT(*) FILTER (WHERE un_medida_capacidade IS NULL) AS un_capacidade_nulos,
    COUNT(*) FILTER (WHERE un_fornecimento IS NULL) AS un_fornecimento_nulos,
    COUNT(*) FILTER (WHERE qt_medicamento IS NULL) AS qt_medicamento_nulos,
    COUNT(*) FILTER (WHERE cnpj_instituicao IS NULL) AS cnpj_instituicao_nulos,
    COUNT(*) FILTER (WHERE cnpj_fornecedor IS NULL) AS cnpj_fornecedor_nulos,
    COUNT(*) FILTER (WHERE cnpj_fabricante IS NULL) AS cnpj_fabricante_nulos
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026;

In [0]:
SELECT
    COUNT(*) AS total_registros,
    COUNT(*) FILTER (WHERE dt_insercao IS NULL) AS dt_insercao_nulos,
    COUNT(*) FILTER (WHERE un_medida_capacidade IS NULL) AS un_capacidade_nulos,
    COUNT(*) FILTER (WHERE un_fornecimento IS NULL) AS un_fornecimento_nulos,
    ROUND(100.0 * dt_insercao_nulos / total_registros, 2) AS percentual_dt_insercao_nulos,
    ROUND(100.0 * un_capacidade_nulos / total_registros, 2) AS percentual_un_capacidade_nulos,
    ROUND(100.0 * un_fornecimento_nulos / total_registros, 2) AS percentual_un_fornecimento_nulos
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026;

In [0]:
SELECT
    ano_compra,
    month(dt_compra) AS mes_compra,
    COUNT(*) AS total_registros
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
WHERE dt_insercao IS NULL
GROUP BY ano_compra, mes_compra
ORDER BY total_registros DESC;

In [0]:
SELECT
    ano_compra,
    month(dt_compra) AS mes_compra,
    COUNT(*) AS total_registros
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
WHERE un_medida_capacidade IS NULL
GROUP BY ano_compra, mes_compra
ORDER BY ano_compra, mes_compra;

In [0]:
SELECT
    co_catmat,
    ds_item
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
WHERE un_medida_capacidade IS NULL
GROUP BY co_catmat, ds_item
ORDER BY co_catmat, ds_item;

In [0]:
SELECT
    ano_compra,
    month(dt_compra) AS mes_compra,
    COUNT(*) AS total_registros
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
WHERE un_fornecimento IS NULL
GROUP BY ano_compra, mes_compra
ORDER BY ano_compra, mes_compra;

In [0]:
SELECT
    co_catmat,
    ds_item
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
WHERE un_fornecimento IS NULL
GROUP BY co_catmat, ds_item
ORDER BY co_catmat, ds_item;

## 2. Consistência

In [0]:
SELECT MIN(dt_compra) AS primeira_compra, MAX(dt_compra) AS ultima_compra
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026;

In [0]:
SELECT ano_arquivo, ano_compra, COUNT(*) AS qtd_diverg_ano_compra_arquivo
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
WHERE ano_compra <> ano_arquivo
GROUP BY ano_arquivo, ano_compra
ORDER BY ano_arquivo, ano_compra

In [0]:
SELECT co_seq_bps, ano_compra, dt_compra
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
WHERE ano_compra IS NOT NULL
  AND dt_compra IS NOT NULL
  AND ano_compra <> EXTRACT(YEAR FROM dt_compra);

In [0]:
SELECT co_seq_bps, dt_compra, dt_insercao,
  date_diff(day, dt_insercao, dt_compra) AS diferenca_dt
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
WHERE dt_compra IS NOT NULL
  AND dt_insercao IS NOT NULL
  AND dt_insercao < dt_compra
ORDER BY dt_compra;

In [0]:
SELECT
    cnpj_instituicao,
    sg_uf,
    no_instituicao,
    COUNT(*) AS qtd_registros
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
WHERE dt_compra IS NOT NULL
    AND dt_insercao IS NOT NULL
    AND dt_insercao < dt_compra
GROUP BY cnpj_instituicao, sg_uf, no_instituicao
ORDER BY qtd_registros DESC;

In [0]:
SELECT *
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
WHERE vl_preco_unitario <= 0

In [0]:
SELECT *
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
WHERE qt_medicamento <= 0

In [0]:
SELECT co_seq_bps, ano_compra, vl_preco_unitario, qt_medicamento, vl_preco_total,
    vl_preco_unitario * qt_medicamento AS total_calculado,
    ABS(vl_preco_total - (vl_preco_unitario * qt_medicamento)) AS diferenca_absoluta
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
WHERE ABS(vl_preco_total - (vl_preco_unitario * qt_medicamento)) > 0.01;

In [0]:
SELECT sg_uf, COUNT(*) AS qtd_diverg_uf
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
WHERE sg_uf IS NOT NULL
  AND UPPER(TRIM(sg_uf)) NOT IN (
      'AC','AL','AP','AM','BA','CE','DF','ES',
      'GO','MA','MT','MS','MG','PA','PB','PR',
      'PE','PI','RJ','RN','RS','RO','RR','SC',
      'SP','SE','TO'
  )
GROUP BY sg_uf
ORDER BY qtd_diverg_uf DESC;

In [0]:
CREATE OR REPLACE FUNCTION fn_valida_cnpj(cnpj_input STRING)
RETURNS BOOLEAN
LANGUAGE PYTHON
AS $$
import re

if cnpj_input is None:
    return None

# Remove caracteres que não sejam números
cnpj = re.sub(r'[^0-9]', '', cnpj_input)

# CNPJ deve possuir 14 dígitos
if len(cnpj) != 14:
    return False

# Rejeita sequências com todos os dígitos iguais
if len(set(cnpj)) == 1:
    return False

# Primeiro dígito verificador
pesos1 = [5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]
soma = sum(int(cnpj[i]) * pesos1[i] for i in range(12))
resto = soma % 11
digito1 = 0 if resto < 2 else 11 - resto

# Compara com o primeiro dígito verificador informado
if digito1 != int(cnpj[12]):
    return False

# Segundo dígito verificador
pesos2 = [6, 5, 4, 3, 2, 9, 8, 7, 6, 5, 4, 3, 2]
soma = sum(int(cnpj[i]) * pesos2[i] for i in range(13))
resto = soma % 11
digito2 = 0 if resto < 2 else 11 - resto

# Compara com o segundo dígito verificador informado
if digito2 != int(cnpj[13]):
    return False

return True
$$;

In [0]:
SELECT cnpj_instituicao, COUNT(*) AS qtd_invalidos
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
WHERE cnpj_instituicao IS NOT NULL
  AND NOT fn_valida_cnpj(cnpj_instituicao)
GROUP BY cnpj_instituicao
ORDER BY qtd_invalidos DESC;

In [0]:
SELECT cnpj_fornecedor, COUNT(*) AS qtd_invalidos
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
WHERE cnpj_fornecedor IS NOT NULL
  AND NOT fn_valida_cnpj(cnpj_fornecedor)
GROUP BY cnpj_fornecedor
ORDER BY qtd_invalidos DESC;

In [0]:
SELECT cnpj_fabricante, COUNT(*) AS qtd_invalidos
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
WHERE cnpj_fabricante IS NOT NULL
  AND NOT fn_valida_cnpj(cnpj_fabricante)
GROUP BY cnpj_fabricante
ORDER BY qtd_invalidos DESC;

In [0]:
-- Verificar melhor forma de fazer esta validação
SELECT
    co_catmat,
    ds_item,
    COUNT(DISTINCT un_fornecimento) AS qtd_un_fornecimento,
    COUNT(DISTINCT vl_capacidade) AS qtd_vl_capacidade,
    COUNT(DISTINCT un_medida_capacidade) AS qtd_un_capacidade
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
GROUP BY co_catmat, ds_item
HAVING COUNT(DISTINCT un_fornecimento) > 1
    OR COUNT(DISTINCT vl_capacidade) > 1
    OR COUNT(DISTINCT un_medida_capacidade) > 1
ORDER BY co_catmat;

## 3. Unicidade

In [0]:
SELECT co_seq_bps, COUNT(*) AS qtd_duplicatas_co_seq_bps
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
GROUP BY co_seq_bps
HAVING COUNT(*) > 1
ORDER BY qtd_duplicatas_co_seq_bps DESC;

In [0]:
SELECT ano_compra, cnpj_instituicao, dt_compra, co_catmat, cnpj_fornecedor, qt_medicamento, vl_preco_unitario, COUNT(*) AS qtd_registros
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
GROUP BY ano_compra, cnpj_instituicao, dt_compra, co_catmat, cnpj_fornecedor, qt_medicamento, vl_preco_unitario
HAVING COUNT(*) > 1
ORDER BY qtd_registros DESC;

## 4. Acurácia

In [0]:
SELECT COUNT(*) AS qtd_divergencias
FROM mvp_puc_engenhariadedados.bronze.bps_2024 b
JOIN mvp_puc_engenhariadedados.silver.bps_2024_a_2026 s
    ON b.co_seq_bps = s.co_seq_bps
WHERE
    b.co_catmat <> s.co_catmat
    OR b.cnpj_instituicao <> s.cnpj_instituicao
    OR b.cnpj_fornecedor <> s.cnpj_fornecedor
    OR b.dt_compra <> s.dt_compra
    OR b.vl_preco_unitario <> s.vl_preco_unitario
    OR b.qt_medicamento <> s.qt_medicamento
    OR b.vl_preco_total <> s.vl_preco_total;

In [0]:
SELECT COUNT(*) AS qtd_divergencias
FROM mvp_puc_engenhariadedados.bronze.bps_2025 b
JOIN mvp_puc_engenhariadedados.silver.bps_2024_a_2026 s
    ON b.co_seq_bps = s.co_seq_bps
WHERE
    b.co_catmat <> s.co_catmat
    OR b.cnpj_instituicao <> s.cnpj_instituicao
    OR b.cnpj_fornecedor <> s.cnpj_fornecedor
    OR b.dt_compra <> s.dt_compra
    OR b.vl_preco_unitario <> s.vl_preco_unitario
    OR b.qt_medicamento <> s.qt_medicamento
    OR b.vl_preco_total <> s.vl_preco_total;

In [0]:
SELECT COUNT(*) AS qtd_divergencias
FROM mvp_puc_engenhariadedados.bronze.bps_2026 b
JOIN mvp_puc_engenhariadedados.silver.bps_2024_a_2026 s
    ON b.co_seq_bps = s.co_seq_bps
WHERE
    b.co_catmat <> s.co_catmat
    OR b.cnpj_instituicao <> s.cnpj_instituicao
    OR b.cnpj_fornecedor <> s.cnpj_fornecedor
    OR b.dt_compra <> s.dt_compra
    OR b.vl_preco_unitario <> s.vl_preco_unitario
    OR b.qt_medicamento <> s.qt_medicamento
    OR b.vl_preco_total <> s.vl_preco_total;

## 5. Outliers

In [0]:
WITH estatisticas AS (
    SELECT co_catmat, COUNT(*) AS qtd_registros,
        PERCENTILE_CONT(0.25)
            WITHIN GROUP (ORDER BY vl_preco_unitario) AS q1,
        PERCENTILE_CONT(0.75)
            WITHIN GROUP (ORDER BY vl_preco_unitario) AS q3
    FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026
    WHERE co_catmat IS NOT NULL
      AND vl_preco_unitario IS NOT NULL
      AND vl_preco_unitario > 0
    GROUP BY co_catmat
    HAVING COUNT(*) >= 5
)

SELECT COUNT(*) FILTER(WHERE vl_preco_unitario < q1 - 1.5 * (q3 - q1)) AS qtd_potenciais_outliers_inferiores,
    COUNT(*) FILTER(WHERE vl_preco_unitario > q3 + 1.5 * (q3 - q1)) AS qtd_potenciais_outliers_superiores
FROM mvp_puc_engenhariadedados.silver.bps_2024_a_2026 b
JOIN estatisticas e
    ON b.co_catmat = e.co_catmat;